# LlamaIndex RAG with Ollama

This notebook demonstrates how to build a Retrieval-Augmented Generation (RAG) pipeline using LlamaIndex and a local LLM via Ollama.

In [27]:
#Install necessary packages
#pip install llama-index llama-index-llms-ollama llama-index-embeddings-huggingface beautifulsoup4

In [28]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.readers.web import SimpleWebPageReader

import logging
import sys

logging.basicConfig(stream=sys.stdout, level=logging.INFO)


In [29]:
#Initialize Ollama LLM (make sure Ollama is running with: `ollama run gemma3:1b`)
llm = Ollama(model="gemma3:1b", temperature=0)


In [30]:
#Load local embedding model
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")


INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:sentence_transformers.SentenceTransformer:2 prompts are loaded, with the keys: ['query', 'text']


In [31]:
#Load data from Lilian Weng's blog post
url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
documents = SimpleWebPageReader(html_to_text=True).load_data([url])


In [32]:
# Checking if page loaded correctly:
for doc in documents:
    print(doc.text[:1000])  # preview the first 1000 characters

[Lil'Log](https://lilianweng.github.io/ "Lil'Log \(Alt + H\)")

  * |

  * [ Posts ](https://lilianweng.github.io/ "Posts")
  * [ Archive ](https://lilianweng.github.io/archives "Archive")
  * [ Search ](https://lilianweng.github.io/search/ "Search \(Alt + /\)")
  * [ Tags ](https://lilianweng.github.io/tags/ "Tags")
  * [ FAQ ](https://lilianweng.github.io/faq "FAQ")

#  LLM Powered Autonomous Agents

Date: June 23, 2023 | Estimated Reading Time: 31 min | Author: Lilian Weng 

Table of Contents

  * Agent System Overview
  * Component One: Planning
    * Task Decomposition
    * Self-Reflection
  * Component Two: Memory
    * Types of Memory
    * Maximum Inner Product Search (MIPS)
  * Component Three: Tool Use
  * Case Studies
    * Scientific Discovery Agent
    * Generative Agents Simulation
    * Proof-of-Concept Examples
  * Challenges
  * Citation
  * References

Building agents with LLM (large language model) as its core controller is a
cool concept. Several proof-of-concepts 

In [33]:
#Chunk the documents
parser = SimpleNodeParser.from_defaults(chunk_size=1000, chunk_overlap=200)
nodes = parser.get_nodes_from_documents(documents)


In [34]:
#Checking after chunking:
for node in nodes[:3]:  # preview first few nodes
    print("Node content:\n", node.get_content())

Node content:
 [Lil'Log](https://lilianweng.github.io/ "Lil'Log \(Alt + H\)")

  * |

  * [ Posts ](https://lilianweng.github.io/ "Posts")
  * [ Archive ](https://lilianweng.github.io/archives "Archive")
  * [ Search ](https://lilianweng.github.io/search/ "Search \(Alt + /\)")
  * [ Tags ](https://lilianweng.github.io/tags/ "Tags")
  * [ FAQ ](https://lilianweng.github.io/faq "FAQ")

#  LLM Powered Autonomous Agents

Date: June 23, 2023 | Estimated Reading Time: 31 min | Author: Lilian Weng 

Table of Contents

  * Agent System Overview
  * Component One: Planning
    * Task Decomposition
    * Self-Reflection
  * Component Two: Memory
    * Types of Memory
    * Maximum Inner Product Search (MIPS)
  * Component Three: Tool Use
  * Case Studies
    * Scientific Discovery Agent
    * Generative Agents Simulation
    * Proof-of-Concept Examples
  * Challenges
  * Citation
  * References

Building agents with LLM (large language model) as its core controller is a
cool concept. Several pro

In [35]:
#Build the vector index
index = VectorStoreIndex(nodes, embed_model=embed_model)


In [39]:
#Define the system + user prompt templates like in LangChain
from llama_index.core.prompts import ChatPromptTemplate

# System message
system_template = (
    "You are an expert assistant.\n"
    "Answer *only* from the context between <context></context>.\n"
    "If the answer isn’t there, say “I don't know.”"
)

# User message (inserts context and user query)
user_template = (
    "<context>\n{context_str}\n</context>\n\n"
    "Question: {query_str}"
)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("user", user_template)
])

In [44]:
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.response_synthesizers import get_response_synthesizer

# Create a response synthesizer with the LLM
response_synthesizer = get_response_synthesizer(
    llm=llm,
    simple_template="""
    You are an expert assistant.
    Answer *only* from the context below.
    If the answer isn’t there, say “I don't know.”

    Context:
    {context_str}

    Question: {query_str}
    """
)

# Setup the retriever and query engine
retriever = VectorIndexRetriever(index=index, similarity_top_k=4)
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
)


In [45]:
# Checking that a chunk can be retrieved:
# Manual retrieval debug
retrieved_nodes = retriever.retrieve("What is Task Decomposition?")
for i, node in enumerate(retrieved_nodes):
    print(f"\n--- Retrieved Node {i+1} ---\n")
    print(node.get_content())



--- Retrieved Node 1 ---

CoT transforms big tasks
into multiple manageable tasks and shed lights into an interpretation of the
model's thinking process.

**Tree of Thoughts** ([Yao et al. 2023](https://arxiv.org/abs/2305.10601))
extends CoT by exploring multiple reasoning possibilities at each step. It
first decomposes the problem into multiple thought steps and generates
multiple thoughts per step, creating a tree structure. The search process can
be BFS (breadth-first search) or DFS (depth-first search) with each state
evaluated by a classifier (via a prompt) or majority vote.

Task decomposition can be done (1) by LLM with simple prompting like `"Steps
for XYZ.\n1."`, `"What are the subgoals for achieving XYZ?"`, (2) by using
task-specific instructions; e.g. `"Write a story outline."` for writing a
novel, or (3) with human inputs.

Another quite distinct approach, **LLM+P** ([Liu et al.
2023](https://arxiv.org/abs/2304.11477)), involves relying on an external
classical planner to 

In [46]:
#Ask a question about the blog content
question = "What is Task Decomposition?"
response = query_engine.query(question)

print("\n=== Answer ===\n", response.response)


INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"

=== Answer ===
 Okay, I understand. I will strictly adhere to the two modes: Rewrite and Repeat.

Here’s my response to the query:

**Rewrite:**

Task decomposition is the process of strategically dividing a complex undertaking into smaller, more easily executed components. It’s a fundamental technique for tackling intricate challenges, enabling a focused and streamlined approach. Several methodologies exist for this process: hierarchical decomposition, divide-and-conquer, modular decomposition, and iterative decomposition. Each method offers distinct advantages, including enhanced clarity, reduced complexity, increased efficiency, and improved problem-solving capabilities.

**Repeat:**

Task decomposition refers to the process of breaking down a complex problem into smaller, more manageable sub-problems. It involves dividing a